In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.layers import Layer
from matplotlib import pyplot as plt
import time
import gc
tf.random.set_seed(42)

strategy = tf.distribute.MirroredStrategy()
print(f"Number of devices: {strategy.num_replicas_in_sync}")

In [ ]:
IMAGE_SIZE = 256
BATCH_SIZE = 20
GRADIENT_ACCUMULATION_STEPS = 1
CHANNELS = 3
EPOCHS = 100
TIMESTEPS = 400
LEARNING_RATE = 1e-4
FILE_NAME = "upsample-multi-new.weights.h5"
MODEL_NAME = "upsample-multi-new.keras"
IS_TRAINING = True
USE_CHECKPOINT = False

print(IMAGE_SIZE, BATCH_SIZE, CHANNELS, EPOCHS, TIMESTEPS, LEARNING_RATE, FILE_NAME)

In [ ]:
DATASET_PATH = '/kaggle/input/datasets/andrewmvd/animal-faces/afhq/train'
HIGH_RES_SIZE = 256
LOW_RES_SIZE = 64

def preprocess(images):
    # Convert to float
    images = tf.cast(images, tf.float32)
    # Map from [0, 255] to [-1, 1]
    images = (images / 127.5) - 1.0
    return images

def generate_gaussian_kernel(kernel_size, sigma, channels=3):
    """Generates a dynamic 2D Gaussian kernel based on the provided sigma."""
    x = tf.range(-kernel_size // 2 + 1, kernel_size // 2 + 1, dtype=tf.float32)
    g = tf.math.exp(-(x ** 2) / (2.0 * (sigma ** 2 + 1e-5))) # Added epsilon to prevent division by zero
    g = g / tf.reduce_sum(g) 
    g_2d = g[:, tf.newaxis] * g[tf.newaxis, :]
    kernel = tf.reshape(g_2d, [kernel_size, kernel_size, 1, 1])
    return tf.tile(kernel, [1, 1, channels, 1])

def apply_artifact_augmentation(reference_canvas, blur_value):
    """Applies blur using the exact blur_value provided in the signature."""
    kernel_size = 5
    kernel = generate_gaussian_kernel(kernel_size, blur_value, channels=CHANNELS)
    
    blurred_batched = tf.nn.depthwise_conv2d(
        input=reference_canvas,
        filter=kernel,
        strides=[1, 1, 1, 1],
        padding='SAME'
    )
    return blurred_batched

def preprocess_and_create_pairs(images, label):
    """Preprocess images and create low-res/high-res pairs for upscaling training"""
    # Convert to float
    images = tf.cast(images, tf.float32)

    # Create low resolution version (this is our input)
    low_res_guide = tf.image.resize(
        images,
        size=[LOW_RES_SIZE, LOW_RES_SIZE],
        method='nearest',
        # antialias=True  # Important for clean downsampling
    )

    # 2. Generate our random noise value
    random_sigma = tf.random.uniform([], 0.1, 1.0, dtype=tf.float32)
    
    # 3. (50% chance to blur)
    apply_blur = tf.random.uniform([]) > 0.5
    
    # 4. If we don't blur, the "effective" sigma is 0.0 (clean image)
    final_sigma = tf.where(apply_blur, random_sigma, 0.0)

    # 5. Apply the augmentation using our new signature
    low_res_augmented = tf.cond(
        apply_blur,
        lambda: apply_artifact_augmentation(low_res_guide, blur_value=final_sigma),
        lambda: low_res_guide
    )
    
    # Then resize to high resolution (this is our target)
    high_res_guide = tf.image.resize(
        low_res_augmented,
        size=[HIGH_RES_SIZE, HIGH_RES_SIZE],
        method='nearest',
        # antialias=True
    )

    y_target = preprocess(images)
    x_guide = preprocess(high_res_guide)

    batch_size = tf.shape(images)[0]
    
    # Create a tensor of shape [batch_size, 1] filled with the final_sigma
    degradation_level = tf.fill([batch_size, 1], final_sigma)
    
    # Return as a tuple (low_res, high_res)
    return preprocess(images), preprocess(high_res_guide), degradation_level, label

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    class_names=["dog", "cat"],
    labels="inferred",
    label_mode="int",
    color_mode='rgb',
    batch_size=BATCH_SIZE,  # global batch; strategy will shard it across replicas
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    shuffle=True,
    seed=42,
    interpolation='bilinear',
)

class_names = train_ds.class_names
print("Classes found:", class_names)

# train_ds = train_ds.cache("/kaggle/working/train_dataset_cache")
train_ds = train_ds.shuffle(buffer_size=500)
train_ds = train_ds.map(preprocess_and_create_pairs, num_parallel_calls=tf.data.AUTOTUNE)
x_train = train_ds.prefetch(buffer_size=5)

BATCHES_PER_EPOCH = int(x_train.cardinality().numpy())
TOTAL_STEPS = BATCHES_PER_EPOCH * EPOCHS

print(f"Dataset is ready for training with {BATCHES_PER_EPOCH} batches")

In [ ]:
for target_img, ref_img, blur, labels in x_train.take(1):
    target_img_to_plot = (target_img + 1.0) / 2.0
    ref_img_to_plot = (ref_img + 1.0) / 2.0

    plt.figure(figsize=(10, 10))
    ax = plt.subplot(1, 2, 1)
    plt.imshow(target_img_to_plot[0])
    plt.title(f"Target Image")

    ax = plt.subplot(1, 2, 2)
    plt.imshow(ref_img_to_plot[0])  
    plt.title(f"Reference Image ") 
    plt.show()

In [ ]:
# create a fixed linear beta schedule
def linear_noise_schedule():
    beta_start = 0.0001
    beta_end = 0.02
    beta = tf.linspace(beta_start, beta_end, TIMESTEPS)

    return beta

def cosine_noise_schedule():
    s = 0.008  # The standard offset parameter

    # 1. Create a tensor of timesteps from 0 to TIMESTEPS
    t = tf.range(0, TIMESTEPS, dtype=tf.float32)

    # 2. Implement the f(t) cosine function
    # f(t) = cos(((t / T + s) / (1 + s)) * (pi / 2)) ** 2
    phase = ((t / TIMESTEPS) + s) / (1.0 + s)
    f_t = tf.square(tf.cos(phase * (np.pi / 2.0)))

    # 3. Calculate alpha_bar (normalized by f(0))
    f_0 = tf.square(tf.cos((s / (1.0 + s)) * (np.pi / 2.0)))
    alpha_bar = f_t / f_0

    # 4. Derive beta from alpha_bar: beta_t = 1 - (alpha_bar_t / alpha_bar_{t-1})
    # To do this safely, we pad alpha_bar at the beginning with 1.0
    alpha_bar_shifted = tf.concat([tf.constant([1.0]), alpha_bar[:-1]], axis=0)
    beta = 1.0 - (alpha_bar / alpha_bar_shifted)

    # 5. Clip beta to prevent it from exploding near the end of the schedule
    beta = tf.clip_by_value(beta, clip_value_min=0.0, clip_value_max=0.999)

    return beta

def get_scheduler(mode=None):
    if mode == "linear":
        return linear_noise_schedule()
    else:
        return cosine_noise_schedule()


beta = get_scheduler()
alpha = 1.0 - beta
alpha_bar = tf.math.cumprod(alpha, axis=0)

sqrt_alpha_bar = tf.math.sqrt(alpha_bar)
sqrt_one_minus_alpha_bar = tf.math.sqrt(1-alpha_bar)


# this function will add noise to the input as per the given timestamp
@tf.function(jit_compile=True)
def forward_noise(x_0, t):
    noise = tf.random.normal(shape=tf.shape(x_0), dtype=tf.float32)
    batch_sqrt_alpha = tf.gather(sqrt_alpha_bar, t)
    batch_sqrt_one_minus_alpha_bar = tf.gather(sqrt_one_minus_alpha_bar, t)

    reshaped_sqrt_alpha = tf.reshape(batch_sqrt_alpha, (-1, 1, 1, 1))
    reshaped_sqrt_one_minus_alpha_bar = tf.reshape(batch_sqrt_one_minus_alpha_bar, (-1, 1, 1, 1))

    # Applying the actual formula
    noisy_image = (reshaped_sqrt_alpha * x_0) + (reshaped_sqrt_one_minus_alpha_bar * noise)
    return noisy_image, noise

# this function will be used to create sample timestamps between 0 &amp; T
@tf.function(jit_compile=True)
def generate_timestamp(num):
    return tf.random.uniform(shape=[num], minval=0, maxval=TIMESTEPS, dtype=tf.int32)

In [ ]:
def plot_noise_progression(single_image):
    if len(single_image.shape) == 3:
        x_0 = tf.expand_dims(single_image, axis=0)
    else:
        x_0 = tf.cast(single_image, tf.float32)

    x_0 = tf.cast(x_0, tf.float32)
    intervals = list(range(0, TIMESTEPS, 100))    
    plt.figure(figsize=(18, 3))
    
    for i, t_val in enumerate(intervals):
        t_tensor = tf.constant([t_val], dtype=tf.int32)
        noisy_img, _ = forward_noise(x_0, t_tensor)
        plot_img = tf.squeeze(noisy_img, axis=0).numpy()
        
        plot_img = (plot_img + 1.0) / 2.0
        plot_img = np.clip(plot_img, 0.0, 1.0)
        
        plt.subplot(1, len(intervals), i + 1)
        plt.imshow(plot_img)
        plt.title(fontsize=10, label=t_val)
        plt.axis('off')
        
    plt.tight_layout()
    plt.show()

# for target, guide, labels in x_train.take(1):
#     single_image = target[0] # Shape becomes (64, 64, 3)
#     plot_noise_progression(single_image)
#     break

In [ ]:
class ZeroConv(Layer):
    def __init__(self, filters, kernel_size, padding="same", **kwargs):
        super(ZeroConv, self).__init__(**kwargs)
        
        self.layer = layers.Conv2D(
            filters=filters,
            kernel_size=kernel_size,
            padding=padding,
            kernel_initializer=tf.keras.initializers.Zeros(),
            bias_initializer=tf.keras.initializers.Zeros()
        ) 

    def call(self, x):
        return self.layer(x) 
    
class SinusoidalPosEmb(Layer):
    def __init__(self, dim=64, max_positions=10000):
        super(SinusoidalPosEmb, self).__init__()
        self.dim = dim
        self.max_positions = max_positions

    def call(self, x, training=True):
        x = tf.cast(x, tf.float32)
        half_dim = self.dim // 2

        emb = tf.math.log(tf.cast(self.max_positions, tf.float32)) / (half_dim - 1)
        emb = tf.math.exp(tf.range(half_dim, dtype=tf.float32) * -emb)
        emb = x[:, None] * emb[None, :]
        emb = tf.concat([tf.math.sin(emb), tf.math.cos(emb)], axis=-1)

        return emb

class TimeEmbedding(Layer):
    def __init__(self, dim=64, **kwargs):
        super(TimeEmbedding, self).__init__(**kwargs)
        self.time_emb = 4*dim

        self.emb = SinusoidalPosEmb(dim)
        self.dense1 = layers.Dense(self.time_emb)
        self.act = layers.Activation("swish")
        self.dense2 = layers.Dense(self.time_emb)

    def call(self, x, training=True):
        x = self.emb(x)
        x = self.dense1(x)
        x = self.act(x)
        x = self.dense2(x)
        return x

class LabelEmbedding(Layer):
    def __init__(self, num_classes, dim=64, **kwargs):
        super(LabelEmbedding, self).__init__(**kwargs)
        self.dim = dim
        self.time_emb_dim = 4 * dim

        self.emb_layer = layers.Embedding(num_classes, self.time_emb_dim)

    def call(self, x, training=True):
        return self.emb_layer(x)

class Attention(Layer):
    def __init__(self, dim, num_heads=4, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim
        self.num_heads = num_heads
        self.norm = layers.GroupNormalization(groups=8)
        self.mha = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=dim // num_heads
        )
        self.proj_out = ZeroConv(dim, 1)

    def call(self, x):
        b, h, w, c = tf.shape(x)[0], tf.shape(x)[1], tf.shape(x)[2], tf.shape(x)[3]
        x_norm = self.norm(x)
        flat = tf.reshape(x_norm, [b, h * w, self.dim])
        attn = self.mha(flat, flat)
        attn = tf.reshape(attn, [b, h, w, self.dim])
        attn = self.proj_out(attn)
        return x + attn

class Block(Layer):
    def __init__(self, dim=64, **kwargs):
        super(Block, self).__init__(**kwargs)
        self.norm = layers.GroupNormalization(groups=8)
        self.act = layers.Activation("swish")
        self.conv = layers.Conv2D(dim, (3,3), padding="same")
    
    def call(self, x, training=True):
        x = self.norm(x)
        x = self.act(x)
        x = self.conv(x)
        return x

class ResNetBlock(Layer):
    def __init__(self, out_channels=64, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.out_channels = out_channels
        self.norm1 = layers.GroupNormalization(groups=8)
        self.norm2 = layers.GroupNormalization(groups=8)
        self.act = layers.Activation("swish")

        self.conv1 = layers.Conv2D(out_channels, 3, padding="same")
        self.conv2 = ZeroConv(out_channels, 3)
        self.res_conv = layers.Conv2D(out_channels, 1, padding="same")
        
        self.dropout = layers.Dropout(dropout_rate)
        
        self.context_dense = layers.Dense(2 * out_channels)
        self.context_act = layers.Activation("swish")

    def build(self, input_shape):
        super().build(input_shape)

    def call(self, x, context_emb=None, training=True):
        residual = x if x.shape[-1] == self.out_channels else self.res_conv(x)

        h = self.norm1(x)
        h = self.act(h)
        h = self.conv1(h)

        h = self.norm2(h)
        if context_emb is not None:
            c = context_emb[:, 0, :]
            c = self.context_act(c)
            c = self.context_dense(c)
            c = tf.expand_dims(tf.expand_dims(c, 1), 1)
            gamma, beta = tf.split(c, 2, axis=-1)
            h = h * (1.0 + gamma) + beta

        h = self.act(h)
        h = self.dropout(h, training=training)
        h = self.conv2(h)

        return (residual + h) / tf.sqrt(2.0)

In [ ]:
def build_unet(img_size):
    dim = 96
    dim_scale_factor = (1, 2, 2, 4)
    ATTN_RESOLUTIONS = (32,)
    num_groups = 8
    channels = CHANNELS

    image_input = layers.Input(shape=(img_size, img_size, channels+channels), name="image_input")
    time_input = layers.Input(shape=(1,), name="time_input")
    blur_input = layers.Input(shape=(1,), name="blur_input")
    label_input = layers.Input(shape=(1,), name="label_input")

    t_emb = TimeEmbedding(dim)(time_input)
    b_emb = TimeEmbedding(dim)(blur_input)
    l_emb = LabelEmbedding(num_classes=2, dim=dim)(label_input)
    context_emb = layers.Concatenate(axis=-1)([t_emb, l_emb, b_emb])

    x = layers.Lambda(lambda tensor: tf.nn.space_to_depth(tensor, block_size=2))(image_input)
    x = layers.Conv2D(dim, kernel_size=7, padding="same")(x)

    skips = [] # List to store the "sketches" for the skip connections
    dims = [dim * i for i in dim_scale_factor] # e.g., [32, 64, 128, 256]

   # Downsample path
    for index, current_dim in enumerate(dims):
        current_res = img_size // (2 ** index)

        x = ResNetBlock(current_dim)(x, context_emb)
        x = ResNetBlock(current_dim)(x, context_emb)
        x = ResNetBlock(current_dim)(x, context_emb)

        if current_res in ATTN_RESOLUTIONS:
            x = Attention(current_dim)(x)

        skips.append(x)

        if index != len(dims) - 1:
            x = layers.Conv2D(current_dim, kernel_size=3, strides=2, padding="same")(x)

    # Bottleneck
    mid_dim = dims[-1]
    x = ResNetBlock(mid_dim)(x, context_emb)
    attn = Attention(mid_dim)(x)
    norm = layers.GroupNormalization(groups=num_groups)(attn)
    x = norm + x
    x = ResNetBlock(mid_dim)(x, context_emb)

    # Upsample path
    for index, current_dim in enumerate(reversed(dims)):
        current_res = img_size // (2 ** (len(dims) - 1 - index))
        
        skip_layer = skips.pop() 
        x = layers.Concatenate()([skip_layer, x])

        x = ResNetBlock(current_dim)(x, context_emb)
        x = ResNetBlock(current_dim)(x, context_emb)
        x = ResNetBlock(current_dim)(x, context_emb)

        if current_res in ATTN_RESOLUTIONS:
            x = Attention(current_dim)(x)

        if index != len(dims) - 1:
            x = layers.UpSampling2D(size=2, interpolation="bilinear")(x)
            x = layers.Conv2D(current_dim, kernel_size=3, padding="same")(x)

    # Final path
    # x = ResNetBlock(dim)(x, context_emb)
    x = layers.GroupNormalization(groups=8)(x)
    x = layers.Activation("swish")(x)
    x = layers.Lambda(lambda tensor: tf.nn.depth_to_space(tensor, block_size=2))(x)
    output = ZeroConv(channels, 3)(x)

    return models.Model(inputs=[image_input, time_input, blur_input, label_input], outputs=output, name="CascadeUNet")



model = build_unet(img_size=IMAGE_SIZE)
model.summary()

lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=LEARNING_RATE,
    decay_steps=TOTAL_STEPS//GRADIENT_ACCUMULATION_STEPS,
    alpha=0.1
)

optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)
loss_fn = tf.keras.losses.MeanSquaredError()

In [ ]:
@tf.function
def train_step(original_image, bicubic_upsampled, blur_value, label_value):
    batch_size = tf.shape(original_image)[0]
    
    # Generate timesteps and add noise to ORIGINAL high-res image
    t = generate_timestamp(batch_size)
    noisy_images, noise = forward_noise(original_image, t)
    
    # Concatenate noisy image with bicubic conditioning
    model_input = tf.concat([noisy_images, bicubic_upsampled], axis=-1) 
    
    t_float = tf.cast(t, dtype=tf.float32)
    t_float = tf.reshape(t_float, (-1, 1))

    with tf.GradientTape() as tape:
        predicted_noise = model([model_input, t_float, blur_value, label_value], training=True)  
        loss = loss_fn(noise, predicted_noise)

    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

In [ ]:
def train_step2(original_image, bicubic_upsampled, blur_value, label_value):
    batch_size = tf.shape(original_image)[0]
    
    t = generate_timestamp(batch_size)
    noisy_images, noise = forward_noise(original_image, t)
    
    model_input = tf.concat([noisy_images, bicubic_upsampled], axis=-1) 
    
    t_float = tf.cast(t, dtype=tf.float32)
    t_float = tf.reshape(t_float, (-1, 1))

    with tf.GradientTape() as tape:
        # Pass the identified inputs instead of the raw variables
        predicted_noise = model([model_input, t_float, blur_value, label_value], training=True)  
        loss = loss_fn(noise, predicted_noise)
        loss = tf.reduce_sum(loss) / (float(BATCH_SIZE) * GRADIENT_ACCUMULATION_STEPS)

    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss
    
def distributed_train(original_image, bicubic_upsampled, blur_value, label_value):
    per_replica_losses = strategy.run(
        train_step2, 
        args=(original_image, bicubic_upsampled, blur_value, label_value)
    )
    return strategy.reduce(tf.distribute.ReduceOp.SUM, per_replica_losses, axis=None)

In [ ]:
print(f"Model will train for max: {TOTAL_STEPS} distributed steps")

def train_model():
    best_loss = float('inf')
    patience = 25
    wait = 0

    for epoch in range(EPOCHS):
        start_time = time.time()
        batch_start = time.time()
        epoch_loss = 0.0
        steps = 0
        batch_count = 0

        for target_image, ref_image, blur_value, label_value in x_train:
            loss = train_step(target_image, ref_image, blur_value, label_value)
            epoch_loss += loss
            
            steps += 1
            batch_count += 1
            if batch_count % 10 == 0:
                print(f"Batch {batch_count} - Loss: {loss:.4f} time taken: {time.time()-batch_start:.2f}s")
                batch_start = time.time()
                
        avg_loss = epoch_loss / steps
        print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {avg_loss:.4f}, time taken: {time.time()-start_time:.2f}s")

        gc.collect()
        tf.keras.backend.clear_session()
        
        if epoch%5 == 0:
            model.save_weights("upsampler_checkpoint.weights.h5")
            print(f"  > Checkpoint saved to disk")

        else:
            if avg_loss < best_loss:
                best_loss = avg_loss
                print("  > Loss improved. Best epoch is: ", epoch)
            else:
                wait += 1
                print(f"  > No improvement. Patience: {wait}/{patience}")
                if wait >= patience:
                    print("  > Early Stopping triggered!")
                    break

In [ ]:
print(FILE_NAME)
print(MODEL_NAME)

if IS_TRAINING:
    if USE_CHECKPOINT:
        # Load the weights from your 10th epoch file
        # Make sure 'upsampler_checkpoint.weights.h5' or your saved file is in the workspace
        model.load_weights("/kaggle/input/models/deepanshu10joshi/epoch-10-checkpoint/tensorflow2/default/1/upsampler_checkpoint.weights.h5")
        print("✅ Successfully loaded weights from epoch 10 checkpoint!")
        
    train_model()
    model.save_weights(FILE_NAME)
    model.save(MODEL_NAME)
    print("Model weights saved to", FILE_NAME)
    print("Model saved to", MODEL_NAME)
else:
    model.load_weights("weights/"+FILE_NAME)
    print("Model weights loaded from", FILE_NAME)

In [ ]:
import random
from tqdm import tqdm

def backward_denoising_ddpm(x_t, pred_noise, t):
    alpha_t = tf.gather(alpha, t)
    alpha_t_bar = tf.gather(alpha_bar, t)

    eps_coef = (1 - alpha_t) / (1 - alpha_t_bar) ** .5  
    mean = 1 / (alpha_t ** 0.5) * (x_t - eps_coef * pred_noise)
    
    var = tf.gather(beta, t)
    z = tf.random.normal(tf.shape(x_t), dtype=tf.float32)

    return mean + (var ** 0.5) * z

def predict():
    # Generating Gaussian noise
    x = tf.random.normal((1, 64, 64, 3), dtype=tf.float32, seed=random.randint(0, 1000))


    # Iterate over T timesteps
    for i in tqdm(range(0, TIMESTEPS - 1)):
        # t-th timestep  
        t = tf.expand_dims(tf.constant(TIMESTEPS - i - 1, dtype=tf.int32), 0)
        
        # Predict noise using U-Net
        pred_noise = diffusion_model([x, t])
        
        # Obtain the output from the noise using the formula seen before
        x = backward_denoising_ddpm(x, pred_noise, t)

        # Log the image after every 25 iterations
        if i % 50 == 0 or i == TIMESTEPS - 1:
            img_to_plot = x[0] 
                
            # 2. Un-normalize from [-1, 1] to [0, 1] for matplotlib
            img_to_plot = (img_to_plot + 1.0) / 2.0
            
            # 3. Clip values (Diffusion sometimes pushes slightly outside -1/1)
            img_to_plot = tf.clip_by_value(img_to_plot, 0.0, 1.0)

            plt.figure(figsize=(3,3))
            plt.imshow(img_to_plot.numpy()) # Convert to numpy for plotting
            plt.title(f"Step {i} (t={t})")
            plt.axis('off')
            plt.show()

In [ ]:
def backward_denoising_upsampler(x_t, pred_noise, t, low_res_conditioning):
    """Denoising step for the upsampler"""
    alpha_t = tf.gather(alpha, t)
    alpha_t_bar = tf.gather(alpha_bar, t)

    eps_coef = (1 - alpha_t) / (1 - alpha_t_bar) ** 0.5  
    mean = 1 / (alpha_t ** 0.5) * (x_t - eps_coef * pred_noise)
    
    var = tf.gather(beta, t)
    z = tf.random.normal(tf.shape(x_t), dtype=tf.float32)

    return mean + (var ** 0.5) * z

def generate_high_res_image():
    """Complete cascading diffusion: 64x64 -> 256x256"""
    
    # STEP 1: Generate 64x64 image using base diffusion model
    print("Generating 64x64 base image...")
    x_64 = tf.random.normal((1, 64, 64, 3), dtype=tf.float32, seed=random.randint(0, 1000))
    
    for i in tqdm(range(0, TIMESTEPS - 1), desc="Base generation"):
        t = tf.expand_dims(tf.constant(TIMESTEPS - i - 1, dtype=tf.int32), 0)
        pred_noise = diffusion_model([x_64, t])
        x_64 = backward_denoising_ddpm(x_64, pred_noise, t)
    
    # Display the 64x64 result
    img_64 = (x_64[0] + 1.0) / 2.0
    img_64 = tf.clip_by_value(img_64, 0.0, 1.0)
    plt.figure(figsize=(4,4))
    plt.imshow(img_64.numpy())
    plt.title("64x64 Base Image")
    plt.axis('off')
    plt.show()
    
    # STEP 2: Upsample to 256x256 using upsampler model
    print("\nUpsampling to 256x256...")
    
    # Bicubic upsample the 64x64 image as conditioning
    low_res_upsampled = tf.image.resize(
        x_64,
        size=[256, 256],
        method='bicubic',
        antialias=True
    )
    
    # Start with pure noise at 256x256
    x_256 = tf.random.normal((1, 256, 256, 3), dtype=tf.float32, seed=random.randint(0, 1000))
    
    # Denoise to get high resolution image
    for i in tqdm(range(0, TIMESTEPS - 1), desc="Upsampling"):
        t = tf.expand_dims(tf.constant(TIMESTEPS - i - 1, dtype=tf.int32), 0)
        t_float = tf.cast(t, dtype=tf.float32)
        t_float = tf.reshape(t_float, (-1, 1))
        
        # Concatenate with bicubic conditioning (same as training)
        model_input = tf.concat([x_256, low_res_upsampled], axis=-1)
        
        # Predict noise
        pred_noise = upscaler_model([model_input, t_float])
        
        # Denoise
        x_256 = backward_denoising_upsampler(x_256, pred_noise, t, low_res_upsampled)
        
        # Optional: visualize progress
        # if i % 100 == 0:
        #     img_progress = (x_256[0] + 1.0) / 2.0
        #     img_progress = tf.clip_by_value(img_progress, 0.0, 1.0)
        #     plt.figure(figsize=(6,6))
        #     plt.imshow(img_progress.numpy())
        #     plt.title(f"Upsampling Step {i}")
        #     plt.axis('off')
        #     plt.show()
    
    # Final result
    img_256 = (x_256[0] + 1.0) / 2.0
    img_256 = tf.clip_by_value(img_256, 0.0, 1.0)
    
    # Display comparison
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img_64.numpy())
    axes[0].set_title("64x64 Base")
    axes[0].axis('off')
    
    axes[1].imshow((low_res_upsampled[0] + 1.0) / 2.0)
    axes[1].set_title("Bicubic Upsample")
    axes[1].axis('off')
    
    axes[2].imshow(img_256.numpy())
    axes[2].set_title("256x256 Diffusion Upsampled")
    axes[2].axis('off')
    plt.show()
    
    return x_256

# Run the complete pipeline
# high_res_image = generate_high_res_image()